# 1 · Getting started — the two front doors to CQL
This is the setup module. It shows the **two ways to run CQL** from a notebook and how they hand
results back to Python and SQL. Everything after this notebook — the
language tour, the clinical measure, the compile artifacts — uses one of these two doors.

1. **`%%cql` — the cell magic.** A cell whose first line is `%%cql` *is* CQL. No quoting,
   no Python wrapper, editor-width room to write real CQL. A cell **declares**, like a Python `def`:
   running it checks the CQL and runs nothing. A cell with no `library` header joins the notebook's
   **session library**, so a later cell sees its defines, as a SQL cell sees a view.
2. **`cql(...)` — the Python API.** A lazy query handle you can inspect, parameterize, reuse and run.
   `cql()` with no argument is the session library, so a Python cell runs what the cells declared.

Write CQL in the magic; run it, and hand results on, from Python. Use `cql("...")` when the CQL is a
value your Python code is manipulating.

**Runs on serverless — nothing to attach, nothing to configure.** Just **Run All**. The one
prerequisite is a workspace where CQL on Databricks is deployed.

## 1. The client
The client, `cql_databricks` — compile requests, result frames, discovery, OAuth and the `%%cql` magic —
comes from this notebook's base environment, **CQL on Databricks**, which the deploy builds with the client
wheel in it. There is nothing to install. For a notebook of your own, pick **CQL on Databricks** under
*Base environment* in the notebook's Environment pane, unless your workspace already makes it the default.

## 2. Load the magic
One cell. `%reload_ext` registers `%%cql` on this notebook's IPython shell, and loads it again when the
cell runs again; the import brings in `cql()`. Everything else — the Compiler
Service endpoint, OAuth, the `cql_*` Unity Catalog function catalog — is discovered on first use.

**Expected output:** nothing. the cell is silent on success; an error here means the notebook is
not on the **CQL on Databricks** base environment.

In [0]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

## 3. Your first CQL cell
The `%%cql` cell declares `Answer`. The Compiler Service checks it, and nothing runs yet: like a
Python `def`, the define is now available, and closing the notebook here would cost nothing more.

The Python cell after it runs it. `cql()` is the session library the cell joined, and `exec()`
compiles it to Spark SQL and runs it on this Spark session.

**Expected output:** the `%%cql` cell prints `added to the session library`. The Python cell shows
one table, one row.

| `__subject_id` | `__val` |
|---|---|
| `*` | `42` |

In [0]:
%%cql
define "Answer": 6 * 7

In [0]:
from cql_databricks.compiler import VALUE_COLUMN, cql

display(cql().exec("Answer"))

## 4. The result shape — always the same two columns
Every define result carries the subject key `__subject_id` and the payload column `__val`.
**A define names the OBJECT — the view or table — never a column**, so you never have to guess
what your result column is called, and two defines always arrive with identical schemas.

The define above declares no `context`, so it resolves to CQL's `Unfiltered` context: there is
no subject, `__subject_id` is the `'*'` seed, and the single row *is* the answer. Notebook 3 shows the same shape with one row per patient.

A **list** is one value like any other, so a list-valued define is still one row per subject — the
whole list arrives as an array in `__val`. (A define whose result is a *stream of resources*, such as
a FHIR retrieve, lowers instead to one row per element plus an `__ordinal` column, because CQL lists
are ordered and the order has to be carried as data. You will see that shape in notebook 2.)

## 5. Hand the result to Python
`exec("BirthYears")` returns an ordinary DataFrame: `.filter()`, `.join()`, `.write` — whatever you
would do with any other frame. That hand-off is the whole integration story, and it is what keeps
CQL from being a walled garden.

**Expected output:**
```
DataFrame -> ['__subject_id', '__val']
BirthYears = [1990, 2010, 1975]
```
Note the order is the order written, not sorted — CQL lists are ordered.

In [0]:
%%cql
define "BirthYears": { 1990, 2010, 1975 }

In [0]:
ages = cql().exec("BirthYears")

print(type(ages).__name__, "->", ages.columns)
print("BirthYears =", ages.collect()[0][VALUE_COLUMN])

## 6. …and `%sql` can read it too
A run of the session library also lands every define as a **session temp view named after the
define**, so a plain SQL cell needs nothing from CQL — it just queries the result. `Answer` from section 3 is still there; the
value inside it is, as always, the column `__val`.

**Expected output:** one row — `subject = *`, `answer = 42`.

In [0]:
%sql
SELECT __subject_id AS subject, __val AS answer FROM Answer

## 7. Choose which defines to evaluate
`exec()` takes the defines to run. One name returns one DataFrame; several return a
`{define: DataFrame}` **dict**. With none, it runs the library's eCQM populations if it has any, else
**every** define.

**Expected output:** `dict_keys(['Screened', 'Unscreened'])`, then `Screened = 840` and
`Unscreened = 360`. `Eligible` is defined and used, but not requested, so it is not returned.

In [0]:
%%cql
define "Eligible":   1200
define "Screened":   840
define "Unscreened": "Eligible" - "Screened"

In [0]:
cohort = cql().exec("Screened", "Unscreened")

print(cohort.keys())
for name, frame in cohort.items():
    print(name, "=", frame.collect()[0][VALUE_COLUMN])

## 8. One session library — later cells see earlier defines
Every cell so far had no `library` header, so each one joined the notebook's **session library**
(`SVC-NOTEBOOK-SESSION-SCOPE`). A later cell refers to an earlier define by name, with nothing to
import. Redefining a name **replaces** it, the way re-running a SQL cell's `CREATE OR REPLACE VIEW`
does, and every define that uses it follows.

Give a cell a `library` header when you want a separate, includable library instead (notebook 5).

**Expected output:** `Total = 1200` (`Screened` + `Unscreened` from section 7). After `Eligible` is
redefined as `1500`, the next run prints `Total = 1500`, and the `%sql` cell reads `total = 1500`:
`Unscreened` is now `660`, so `Total` moved with it.

In [0]:
%%cql
define "Total": "Screened" + "Unscreened"

In [0]:
print("Total =", cql().exec("Total").collect()[0][VALUE_COLUMN])

In [0]:
%%cql
define "Eligible": 1500

In [0]:
# The views are re-created by a run, so run again before SQL reads them.
print("Total =", cql().exec("Total").collect()[0][VALUE_COLUMN])

In [0]:
%sql
SELECT __val AS total FROM Total

## 9. CQL as a Python value
`cql(source)` returns a **lazy query handle**. Nothing runs until you ask it for something, and the
compile is memoized — inspecting a library and then running it costs one round trip, not two.

A string with no `library` header joins the same session library as the cells above. `cql()` checks
at once that the whole library still compiles. `exec()` with no targets runs only the string's own
defines, and `defines()` lists the whole library.

This is the door to use when the CQL is a *value* your code holds: read from a file, resolved from a
registry, or looped over.

**Expected output:**
```
its defines : ['Answer2', 'Doubled']
library     : ['Answer', 'Answer2', 'BirthYears', 'Doubled', 'Eligible', 'Screened', 'Total', 'Unscreened']
populations : []
Doubled     = 84
```
`populations` is empty because this library has no well-known population define, such as
`Initial Population`. Notebook 3 shows one that does.

In [0]:
q = cql('''
define "Answer2": 6 * 7
define "Doubled": "Answer2" * 2
''', spark=spark)

print("its defines :", q.default_targets())
print("library     :", sorted(q.defines()))
print("populations :", q.populations())
print("Doubled     =", q.exec("Doubled").collect()[0][VALUE_COLUMN])

## 10. Parameters — the same handle, specialized
A CQL `parameter` with a `default` is the mechanism every measure uses for its measurement period.
The handle is **immutable-fluent**: `with_parameter` returns a NEW handle, so one bound library can
be reused and specialized rather than rebound.

An override is CQL *expression text*, so a Python string passes through verbatim as CQL — which is
how a date, interval or quantity literal is bound (`"@2026-01-01"`, `"40 'mg'"`).
`with_measurement_period("2026-01-01", "2026-12-31")` is the shorthand for the ubiquitous one, and it
widens both bounds to instants so they compare at the precision real resource
timestamps carry — a detail notebook 2 shows the teeth of.

**Expected output:**
```
default threshold (10) : True
bound threshold  (100) : False
```

In [0]:
scored = cql('''
parameter "Threshold" default 10
define "OverThreshold": 42 > "Threshold"
''', spark=spark)

print("default threshold (10) :", scored.exec("OverThreshold").collect()[0][VALUE_COLUMN])
print("bound threshold  (100) :",
      scored.with_parameter("Threshold", 100).exec("OverThreshold").collect()[0][VALUE_COLUMN])

## 11. Libraries from a folder — the Git Folder setup
Everything so far ran CQL you handed over as *text*. A real measure is not text you paste: it is a
library that `include`s one or more others, published and versioned in a repo.

The compile service resolves **none** of those includes — deliberately. It is a dumb endpoint: the
caller resolves the closure and ships every dependency, and the service only merges what it is
handed. It reads no filesystem, no registry and no network. So the resolver
lives on the client, next to the files.

`library_path` mounts one or more directories of `.cql` as the session's library source, and a
measure in them runs **by name**. In a real deployment that
directory is a **Databricks Git Folder** — your measures repo, synced by the platform — or a Unity
Catalog Volume. It is an ordinary directory either way, so there is no platform code involved:

```python
client = connect(spark, library_path="/Workspace/Users/me/my-measures/cql")
```

Roots are **ordered and earlier wins**, so a work-in-progress directory shadows a vendored content
repo: `library_path=["/Workspace/Users/me/wip", "/Volumes/acme/content/measures"]`.

Here we point it at the `measures/` folder published beside this notebook — the content notebook 3
runs for real. The path is derived from this notebook's own location, so the cell works unchanged
wherever the folder was copied to.

**Expected output:**
```
library_path: /Workspace/…/examples/measures
  FHIRHelpers|4.0.1
  HypertensionBPControl|1.0.0
  MeasureCommon|1.0.0
populations of HypertensionBPControl: ['Initial Population', 'Denominator', 'Denominator Exclusions', 'Numerator']
```
That last line is the whole feature. Binding by name resolved the measure, walked its `include`
closure to find the helper libraries, and shipped all of them to the compiler — and **no data was
touched**, because resolving and compiling a measure is not the same act as running it.

In [0]:
import pathlib

from cql_databricks.compiler import connect

_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
MEASURES = (pathlib.Path("/Workspace") / pathlib.PurePosixPath(_nb).parent.relative_to("/")
            / "measures")
if not MEASURES.is_dir():
    raise FileNotFoundError(f"no measures folder at {MEASURES} — it is published beside this notebook")

client = connect(spark, library_path=str(MEASURES))
print("library_path:", MEASURES)
for ref in client.libraries("*"):
    print(" ", ref)

measure = client.cql(library="HypertensionBPControl").with_measurement_period("2026-01-01", "2026-12-31")
print("populations of HypertensionBPControl:", measure.populations())

## 12. What did *not* happen
No endpoint. No token. No cluster. No `%scala` cell. No catalog or schema named anywhere in this
notebook. A single deployment matching the naming convention resolves itself:

| What | How it was resolved |
|---|---|
| Compiler Service endpoint | looked up in the workspace App registry by name |
| Authentication | your ambient identity, exchanged for an App token (RFC 8693) |
| The `cql_*` operator catalog | read out of `system.information_schema.routines` |
| FHIR data / value sets | the default environment: the catalog tagged `cql_databricks_environment_default`, whose `data` views and `knowledge.valueset` view are read by name — only when the CQL retrieves data |

Each is overridable (`connect(app_name=…, environment=…, data=…, terminology=…)`) — but you have to *want*
to.

The library source is the one thing you *do* name, and that is deliberate: which measures you mean
is a decision, not a convention.

---

### Where next
- **02 · CQL language tour** — what the language can express: null handling, precision dates, UCUM
  units, timing relations, equality vs equivalence, terminology filtering, polymorphic choice types.
- **03 · A clinical quality measure** — that same folder run for real, over FHIR data, producing a
  care-gap report and a rate.
- **04 · Compilation and inspection** — the artifacts a compile produces before anything runs.

In [0]:
# Every cell above ran, so the notebook is green. There are no value assertions here by design: the
# narrative states each expected result, and a compile or runtime failure fails its own cell loudly
# rather than being collected into a summary at the end.
import json
dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "01_getting_started"}))